# Lab 13: AI APIs as Data Tools

*INFO 4617/5617 · Week 13 · Chapter 13, [AI APIs as Data Tools](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-13-ai-apis.html)*

This week every class is live coding: one lab in three parts, one a day. Wednesday's class was cancelled, so the parts run on Monday, November 9, Friday, November 13, and Monday, November 16. Each step below says what we're building and why, and its code cell holds only the comments that frame it: we write the code live, in class. Nothing here is turned in. After each step, a **💬 Discuss** box holds questions to talk over before the next one; click it to open it.

Parts 1 and 2 work on the City of Boulder's revenue reports from week 9, the PDFs whose last page is a picture of a table. Part 3 moves to Wikipedia:
- **Part 1, Monday, November 9:** **document understanding**, reading a whole PDF, pictures of tables included. [Datalab](https://www.datalab.to)'s API turns it into Markdown a program can read, one page first and then the whole report, and we check what it read;
- **Part 2, Friday, November 13:** **a model as a function**. Through [OpenRouter](https://openrouter.ai), a language model reads one short sentence, then one page, and returns JSON in a shape we define, and we check its numbers against week 9's;
- **Part 3, Monday, November 16:** **search by meaning**. Embeddings, lists of numbers that stand for what a passage is about, measure how the opening paragraphs of five Wikipedia articles about politicians have changed since 2004. We start small: one old version by hand, and one embedding request on three short sentences, before all 116 leads. If there's time, an optional last step shows **tools**: a model reads the tools that the Census Bureau's MCP server offers, and writes one request to week 11's API, which our code sends and checks.

No chatbots: every call to a model is one request in and one JSON answer out, and every answer gets checked. Along the way, short ⏱ microsprints ask you to change one thing, alone or in pairs, and share what you find.

**Before class:** open this notebook in the `webdata` environment (chapter 1), and have your email address ready for the User-Agent. You need **two free keys**, and Part 3 also needs week 11's, each stored as an environment variable by one of week 11's three safe ways (set it, close the terminal, open a new one, and start Jupyter from there):
- `DATALAB_KEY`: sign up at [datalab.to](https://www.datalab.to), then copy a key from [API Keys](https://www.datalab.to/app/keys). The free plan includes a monthly allowance of at least $10 ([pricing](https://www.datalab.to/pricing)); this lab spends about 5 cents of it;
- `OPENROUTER_KEY`: sign up at [openrouter.ai](https://openrouter.ai), then create a key at [Keys](https://openrouter.ai/settings/keys). The models we use end in `:free`: they cost nothing, up to a daily limit;
- `CENSUS_KEY`, for Part 3's Step 14: week 11's key from the [Census Bureau](https://api.census.gov/data/key_signup.html). If week 11's lab could read it, so can this one.

Treat each key like a password: never paste it into a cell or a code file. On Friday and on Monday, November 16, start at that day's setup step. Don't run every cell above: that would pay Datalab again.

# Part 1 — Monday: from a PDF to text a program can read

In week 9, pdfplumber read the text layer of Boulder's revenue reports exactly. It stopped at Exhibit 3 of the December 2024 report, a picture of a table, beside a note asking readers to phone the city for "a screen readable version". Today a **document-understanding** API, one that reads a page's pictures as well as its text, reads that page first, on its own, and then the whole report. Then we check what it read, the way week 9 checked pdfplumber.

## Step 0 — Set up, and check your keys (4 minutes)

Both cells are given. Put your email in the first cell's `HEADERS`, and run it: it imports what we need and prints whether each key is set, never the key itself (`"set" if ... else "MISSING"` picks one of the two words). If one says `MISSING`, quit Jupyter, open a new terminal, and start Jupyter again from there. If you can't restart now, the second cell asks for the Datalab key with [`getpass()`](https://docs.python.org/3/library/getpass.html), which doesn't show what you type. Run it either way: it makes `datalab_headers`, a copy of `HEADERS` with the key added, and sets today's addresses. The chapter's ["Two Keys"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-13-ai-apis.html#two-keys) has the steps to get the keys.

In [ ]:
# Given: Part 1's setup. Put your email in HEADERS, then run it. It imports what Part 1 needs, and prints
# whether each key is set or MISSING, never the key itself
import os
import re
import time
import requests

HEADERS = {"User-Agent": "WebDataScience/1.0 (your-email@colorado.edu)"}
print("DATALAB_KEY:", "set" if os.environ.get("DATALAB_KEY") else "MISSING")
print("OPENROUTER_KEY:", "set" if os.environ.get("OPENROUTER_KEY") else "MISSING")

In [ ]:
# Given: run this whatever the check said. If DATALAB_KEY was MISSING, it asks for it with getpass, in a box under
# this cell. Then it reads the key; datalab_headers is a copy of HEADERS (dict(HEADERS)) with the key added, for
# every request to Datalab; and the addresses: Datalab's API, week 9's PDFs (book), and their saved Markdown (saved)
from getpass import getpass
if not os.environ.get("DATALAB_KEY"):
    os.environ["DATALAB_KEY"] = getpass("DATALAB_KEY isn't set. Paste it here: ")
datalab_key = os.environ["DATALAB_KEY"]
datalab_headers = dict(HEADERS)
datalab_headers["X-API-Key"] = datalab_key
datalab_base = "https://www.datalab.to/api/v1"
book = "https://raw.githubusercontent.com/cuinfoscience/Web-Data-Science-Book/main/data/ch-09/"
saved = "https://raw.githubusercontent.com/cuinfoscience/Web-Data-Science-Book/main/data/ch-13/"

<details>
<summary>💬 Discuss</summary>

- Census, FRED and YouTube gave you keys to read their data; Datalab and OpenRouter read whatever you send them. What wouldn't you send?
- Datalab bills by the page, after a free allowance. What does that allowance let the company learn about you?

</details>

## Step 1 — One page first (12 minutes)

Datalab's [Convert API](https://documentation.datalab.to/docs/recipes/conversion/conversion-api-overview) takes a PDF and returns Markdown. A given cell holds the settings: Markdown, in `balanced` mode, with each page marked (`paginate`), and converted afresh ([`skip_cache`](https://documentation.datalab.to/api-reference/convert-document)), so the cost we see is real. Before the whole report, convert one page: page 15, Exhibit 3, the picture of a table. Open page 15 of the PDF beside you. `dict(settings)` copies `settings`; to the copy, add `file_url`, the address Datalab downloads the PDF from, and `page_range`, which counts pages from 0, so page 15 is `"14"`.

Datalab answers at once with a **ticket**, `request_check_url`, the address where the result will be. Ask it until its `status` says `complete`. You should see 1 page, its cost in cents (the whole report's 15 pages cost 4.5), and a table whose rows give a tax, a year, twelve months and a total.

In [ ]:
# Given: how Datalab should convert: to Markdown, in balanced mode, with each page marked, and afresh,
# not from its cache
settings = {
    "output_format": "markdown",
    "mode": "balanced",
    "paginate": "true",
    "skip_cache": "true",
}

In [ ]:
# one_page: a copy of settings (dict(settings)); add "file_url", book + "revenue-report-2024-12.pdf",
# and "page_range", "14" (page 15: page_range counts from 0)
# Your code here

In [ ]:
# one_job: the JSON of a POST to f"{datalab_base}/convert", with data=one_page and headers=datalab_headers; print it
# Your code here

In [ ]:
# one_check: the JSON of a GET to one_job["request_check_url"], with headers=datalab_headers; print its "status"
# Run this cell again until the status is complete
# Your code here

In [ ]:
# Print one_check's "page_count" and "total_cost" (in cents); then its "markdown" from the first "|", the table
# Your code here

<details>
<summary>💬 Discuss</summary>

- Every API so far answered with the data. Why might a service answer with a ticket instead?
- Typed as `"file"`, the key for uploading a file, rather than `"file_url"`, this request got HTTP 422 with `'loc': ['body', 'file']` and `'msg': "Value error, Expected UploadFile, received: <class 'str'>"`. What do `loc` and `msg` say went wrong, and why is the first line to fail the next cell's `one_job["request_check_url"]`?
- Print `one_markdown[:60]`. Is the number in the page marker the page's number in the PDF, or `page_range`'s?
- Your table's first row matches the PDF. What would you check before you trusted the other 29?

</details>

## Step 2 — The whole report, and wait for it (7 minutes)

Now all 15 pages: the same settings, with `file_url` and no `page_range`, sent the same way. Asking again and again until a job is done is called **polling**. You did it by hand in Step 1; now a given cell asks the [result check](https://documentation.datalab.to/api-reference/convert-result-check) every two seconds while the status is `processing`, the way week 12's loop followed the `cursor` until there wasn't one. It waits only while `processing`, not until `complete`, because a `failed` job never becomes `complete`. Then it prints `error`. Read it even when the status is `complete`: a job for an address with no PDF behind it came back `complete`, with a 404 in `error`. Then it prints how many pages Datalab read, `page_count`, and what they cost, `total_cost`, in cents.

In [ ]:
# data: a copy of settings, with "file_url" the same report's address, and no "page_range"
# Your code here

In [ ]:
# job: send data as you sent one_page in Step 1; print it
# Your code here

In [ ]:
# Given: ask the ticket's address once, then every two seconds while the job is processing, as you did by hand
# in Step 1. Then print its status and error, and what it read, cost and took
ticket = job["request_check_url"]
check = requests.get(ticket, headers=datalab_headers).json()
while check["status"] == "processing":
    time.sleep(2)
    check = requests.get(ticket, headers=datalab_headers).json()
print("status:", check["status"])
print("error:", check.get("error"))
print(check["page_count"], "pages")
print(check["total_cost"], "cents")
print(round(check["runtime"], 1), "seconds")

<details>
<summary>💬 Discuss</summary>

- The loop stops on `failed` as well as `complete`, and prints the `error`. If a job stayed `processing` and never finished, what would your notebook do, and how does the chapter's loop guard against that?
- Fifteen pages cost a few cents. About what would every year-end report Boulder has published cost?
- When would you choose `lite` mode, which cost the chapter about an eighth as much for this report?

</details>

## Step 3 — Read the Markdown, page by page (10 minutes)

The document is in `check["markdown"]`, one long string. With `paginate`, each page starts with a **page marker**: a number in braces, such as `{14}`, then a line of dashes. A given function, `split_pages()`, splits the string at each marker with [`re.split()`](https://docs.python.org/3/library/re.html#re.split), and drops the empty piece before the first. Its pattern is a **regular expression**, a short way to describe text: `\{\d+\}` matches a number in braces, and `-{10,}` ten or more dashes. Python counts from 0, so page 2 is `pages[1]`. Print page 2, the summary table week 9 read with pdfplumber; then Exhibit 3, page 15, from its first `|`, 150 characters a line. **Never print page 1:** the cover gives a staff member's email and phone number, and Exhibit 3's note the tax office's, and we keep contact details out of our outputs, as week 9 blacked them out. See the chapter's ["Reading the Markdown"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-13-ai-apis.html#reading-the-markdown).

In [ ]:
# Given: split_pages(markdown) splits Datalab's Markdown at each page marker (a number in braces, then a line of
# dashes, described by a regular expression), and returns the pages, without the empty piece before the first
def split_pages(markdown):
    pieces = re.split(r"\n*\{\d+\}-{10,}\n*", markdown)
    return pieces[1:]

In [ ]:
# markdown: check["markdown"], the whole report; pages: split_pages(markdown)
# Print how many pages, then the first 1,200 characters of pages[1], the summary. Never pages[0]
# Your code here

In [ ]:
# exhibit: pages[14] from its first "|" (the table, below the note); print the first 150 characters of its first 10 lines
# Your code here

### ⏱ Microsprint (4 minutes, alone or in pairs)

Datalab doesn't only read text: where a page has a chart, it writes a description of it. Chart 8, on page 9, is a pie chart of marijuana revenue, with its figures printed on the slices.

**Do:** print page 9 (`pages[8]`), and compare the description under the chart with the figures in its table.
**Share:** one sentence of the description that the figures contradict, and who wrote that sentence: the city, or Datalab?

In [ ]:
# Microsprint: print pages[8]; read the description under Chart 8 against its figures
# Your code here

<details>
<summary>💬 Discuss</summary>

- The city's note asks readers to phone for a screen readable version of Exhibit 3. Who would have to check Datalab's table before the city could publish it as that version?
- The description of Chart 8 calls one slice the largest, then gives a larger one. How would you tell what a model wrote on these pages from what it read off them?
- Page 10's bar chart of sugar-sweetened beverage tax has no figures printed on it. Print `pages[9]`: where could the numbers Datalab wrote beside it have come from?

</details>

## Step 4 — Check what it read (13 minutes)

Week 9's first check was a sum: a row's months should add up to its total ([chapter 9, "Checking an Extraction"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-09-pdfs.html#checking-an-extraction)). In the Markdown, each row of the table is one line, with its cells between `|` characters. Take one row by hand: line 3 of `exhibit`, Retail Sales Tax for 2019. [`split("|")`](https://docs.python.org/3/library/stdtypes.html#str.split) gives its cells: `cells[2]` is the year, `cells[3:15]` the twelve months, and `cells[15]` the total. Then a given function, `check_sums(page)`, does that for every row of a page. It takes out the `<b>` tags and the commas, keeps a row whose third cell is a year ([`isdigit()`](https://docs.python.org/3/library/stdtypes.html#str.isdigit)), adds its months, and prints any row that misses its total by more than $5, as week 9 allowed for rounding.

In [ ]:
# exhibit_lines: exhibit's lines (splitlines()); row: exhibit_lines[3], Retail Sales Tax for 2019
# Print row's first 150 characters
# Your code here

In [ ]:
# cells: row split on "|"; print the year (cells[2]) and the total (cells[15]); then the twelve months (cells[3:15])
# Your code here

In [ ]:
# Given: check_sums(page) does that for every row of a page. It takes out the <b> tags and commas, keeps a row whose
# third cell is a year, adds its twelve months, and prints the row if the sum misses its total by more than $5
def check_sums(page):
    checked = 0
    for line in page.splitlines():
        line = line.replace("<b>", "").replace("</b>", "")
        line = line.replace(",", "")
        cells = line.split("|")
        if len(cells) > 16 and cells[2].strip().isdigit():
            months = 0
            for month in cells[3:15]:
                months = months + int(month)
            checked = checked + 1
            if abs(months - int(cells[15])) > 5:
                print(cells[2], months, cells[15])
    print(checked, "rows checked")

In [ ]:
# Check every row of Exhibit 3: check_sums(pages[14])
# Your code here

### ⏱ Microsprint (5 minutes, alone or in pairs)

The 2018 report is older: its Exhibit 3 is a table of text, not a picture. Datalab converted it the same way on 2026-11-07, and the book saved the Markdown, so checking it costs nothing.

**Do:** read the book's saved `revenue-report-2018.md` (`saved` + its name), split it into pages, and run `check_sums()` on its page 15.
**Share:** how many rows the check found, and how many missed.

In [ ]:
# Microsprint: md_2018: the .text of a GET to saved + "revenue-report-2018.md", with HEADERS;
# pages_2018: split_pages(md_2018); then check_sums(pages_2018[14])
# Your code here

<details>
<summary>💬 Discuss</summary>

- Every row added up. What can a sum not catch? (Week 9's Docling put the right figures beside the wrong label.)
- Week 9 ran Docling on a laptop, free, in about a minute; Datalab took seconds and cost cents. What else differs between the two if the PDF holds a student's records rather than a city's?
- Our code assumes every request worked. If Datalab answered 401 (Unauthorized) because the key was wrong, which line would fail first?

</details>

## Step 5 — Save the text for Part 2 (3 minutes)

Write the 2024 report's Markdown to a file, so Part 2 doesn't have to pay to read it again. (Part 2 also works from the book's saved copies of all seven year-end reports, made the same way on 2026-11-07, so a student who missed today can start there.)

In [ ]:
# md_path: "revenue-report-2024-12.md"; write markdown, the 2024 report's, to it,
# with open(md_path, "w", encoding="utf-8") as f
# Your code here

<details>
<summary>💬 Discuss</summary>

- We saved the Markdown, not what Datalab was asked. What else would you write down, so that someone could make this file again and get the same text?
- Part 2 reads the book's copy, made on 2026-11-07, not yours. What could differ between the two, and how would you check?

</details>

## End of Part 1

Wednesday's class was cancelled. On Friday, November 13, Part 2 hands one page of this Markdown to a language model and asks for JSON in a shape we define: the summary table's totals. Then we check the model's numbers against week 9's, for seven years of reports. It starts with its own setup step, so you don't need today's cells to have run.

# Part 2 — Friday, November 13: a model as a function

A **language model**, the kind of program behind a chatbot, can be called like any other API: we send it an instruction and some text, and it returns an answer. We won't chat with it. We ask for **structured output**, JSON in a shape we define, so the answer goes straight into a DataFrame. We start with one short sentence whose answer we know, and only then send a page. Then we check the answer, because a model's numbers look right whether or not they are.

## Step 6 — Set up again (4 minutes)

Part 2 starts fresh, so a student who missed Monday can start here. Both cells are given. Put your email in the first cell's `HEADERS`, and run it: it imports what we need and checks `OPENROUTER_KEY`. Run the second whatever the check said. It asks for the key if it's missing; makes `openrouter_headers`, a copy of `HEADERS` with the key added in an `Authorization` header; sets two addresses, OpenRouter's API and the book's saved Markdown of the seven year-end reports (`data/ch-13/`), made as on Monday; and defines Monday's `split_pages()` again. [OpenRouter](https://openrouter.ai/docs/quickstart) takes one kind of request and passes it to models from many companies. Models whose ID ends in `:free` cost nothing, but they're shared, so they're sometimes busy.

In [ ]:
# Given: Part 2's setup. Put your email in HEADERS, then run it. It imports what Part 2 needs, and prints
# whether OPENROUTER_KEY is set or MISSING, never the key itself
import os
import re
import json
import time
import requests
import pandas as pd

HEADERS = {"User-Agent": "WebDataScience/1.0 (your-email@colorado.edu)"}
print("OPENROUTER_KEY:", "set" if os.environ.get("OPENROUTER_KEY") else "MISSING")

In [ ]:
# Given: run this whatever the check said. If OPENROUTER_KEY was MISSING, it asks for it with getpass, in a box under
# this cell. Then it reads the key; openrouter_headers is a copy of HEADERS with the key added, in an Authorization
# header; the addresses of OpenRouter's API and of the book's saved Markdown; and Monday's split_pages(), again
from getpass import getpass
if not os.environ.get("OPENROUTER_KEY"):
    os.environ["OPENROUTER_KEY"] = getpass("OPENROUTER_KEY isn't set. Paste it here: ")
openrouter_key = os.environ["OPENROUTER_KEY"]
openrouter_headers = dict(HEADERS)
openrouter_headers["Authorization"] = f"Bearer {openrouter_key}"
openrouter_base = "https://openrouter.ai/api/v1"
saved = "https://raw.githubusercontent.com/cuinfoscience/Web-Data-Science-Book/main/data/ch-13/"


def split_pages(markdown):
    pieces = re.split(r"\n*\{\d+\}-{10,}\n*", markdown)
    return pieces[1:]

<details>
<summary>💬 Discuss</summary>

- OpenRouter stands between you and the companies that run the models. What does that make easier?
- With OpenRouter in the middle, who sees each page you send?

</details>

## Step 7 — Find the summary page (5 minutes)

Read the 2024 report's Markdown from the saved copy, and split it into pages as on Monday. Then keep the first page that holds the table's last row, `Total Sales and Use Tax`. As a pandas Series, the pages can say which of them contain it, with [`str.contains()`](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.contains.html) (`regex=False` reads the phrase as plain text, not a regular expression); `.iloc[0]` takes the first that does. That one page is all the model will see. It's about 1,500 **tokens**. A token is a word or a piece of one: the unit a model reads and writes text in, and is billed by.

In [ ]:
# markdown: the .text of a GET to saved + "revenue-report-2024-12.md", with HEADERS; pages: split_pages(markdown)
# Your code here

In [ ]:
# pages_series: pd.Series(pages); has_total: whether each page contains "Total Sales and Use Tax" (regex=False)
# summary: the first page that does, pages_series[has_total].iloc[0]; print its first 1,000 characters
# Your code here

<details>
<summary>💬 Discuss</summary>

- The phrase is on two pages of every report, and we took the first. What would the model see if that page held a different table?
- What would the model miss if the table ran onto the next page?

</details>

## Step 8 — One sentence, then one page, as JSON (21 minutes)

A request to [chat completions](https://openrouter.ai/docs/api/reference/overview) holds a list of `messages`. We build it in three rounds, each a small change to the last:
1. **One sentence, plain.** One `user` message: a sentence whose answer we know, `173908431`. We send a **fallback list** of free `models`: if the first is busy, OpenRouter tries the next ([model fallbacks](https://openrouter.ai/docs/guides/routing/model-fallbacks)). Mind the s: with `model`, OpenRouter answers from your account's default model, which may not be free. **Temperature** sets how much chance goes into each word; `temperature` 0 asks for the most likely answer. Print `error` first: `None` if the request worked. The answer is in `choices[0]["message"]["content"]`, as text: [`repr()`](https://docs.python.org/3/library/functions.html#repr) shows its quotes.
2. **One field, as JSON.** `response_format` asks for [structured outputs](https://openrouter.ai/docs/guides/features/structured-outputs): an answer that fits a **JSON Schema**, a description, in JSON, of the fields an answer must have and the type of each. `ONE_FIELD` has one, `total_current`, an integer. `provider` `{"require_parameters": True}` skips any endpoint that doesn't support every parameter we send ([provider routing](https://openrouter.ai/docs/guides/routing/provider-selection#requiring-providers-to-support-all-parameters)). [`json.loads()`](https://docs.python.org/3/library/json.html#json.loads) turns the text into a dictionary. You should see `173908431` again, now a number.
3. **The page, four fields.** A `system` message holds the instruction, and the `user` message the page. `usage` counts the tokens, including `reasoning_tokens`, text some models write to work an answer out, which isn't part of it.

See the chapter's ["One Request, One JSON Object"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-13-ai-apis.html#one-request-one-json-object).

In [ ]:
# Given: three free models, tried in this order (checked on 2026-11-16; free models come and go, so check
# OpenRouter's list, https://openrouter.ai/models?max_price=0, before class)
MODELS = [
    "nvidia/nemotron-3.5-super-vl:free",
    "google/gemma-4-31b-it:free",
    "dots-studio/dots-3-note-preview:free",
]

In [ ]:
# sentence: "The 2024 total was $173,908,431. Write that number as digits only."
# Your code here

In [ ]:
# plain_body: "models": MODELS (with an s), "temperature": 0, and "messages": a list of one message,
# {"role": "user", "content": sentence}
# Your code here

In [ ]:
# plain: the JSON of a POST to f"{openrouter_base}/chat/completions", with json=plain_body and
# headers=openrouter_headers. Print plain.get("error"): None if it worked
# Your code here

In [ ]:
# Print the model that answered (plain["model"]) and whether it's in MODELS
# plain_text: the first choice's message's "content"; print repr(plain_text), which shows its quotes
# Your code here

In [ ]:
# Given: a JSON Schema with one field, total_current, a whole number (an integer); and the response_format that
# asks for an answer in its shape, "strict": no other fields
ONE_FIELD = {
    "type": "object",
    "properties": {"total_current": {"type": "integer"}},
    "required": ["total_current"],
    "additionalProperties": False,
}
one_format = {"type": "json_schema", "json_schema": {"name": "total", "strict": True, "schema": ONE_FIELD}}

In [ ]:
# one_body: a copy of plain_body (dict(plain_body)); set its "response_format" to one_format,
# and its "provider" to {"require_parameters": True}
# Your code here

In [ ]:
# one: send one_body as you sent plain_body; print one.get("error")
# Your code here

In [ ]:
# one_text: the first choice's message's "content", still text; print the model that answered, and one_text
# one_answer: json.loads(one_text), a dictionary; print whether its "total_current" == 173908431
# Your code here

In [ ]:
# Given: the whole answer's shape, as a JSON Schema: four fields, each with a type and a description; the
# instruction, for the system message; and response_format, which asks for SCHEMA's shape as one_format asked for ONE_FIELD's
MONTHS = ["January", "February", "March", "April", "May", "June", "July",
          "August", "September", "October", "November", "December"]
SCHEMA = {
    "type": "object",
    "properties": {
        "month": {"type": "string", "enum": MONTHS,
                  "description": "The month the year-to-date figures run through"},
        "year": {"type": "integer",
                 "description": "The report's current year: the later of the two years the table compares"},
        "total_prior": {"type": "integer",
                        "description": "The table's last row, Total Sales and Use Tax, for the earlier year, in dollars"},
        "total_current": {"type": "integer",
                          "description": "The table's last row, Total Sales and Use Tax, for the report's current year, in dollars"},
    },
    "required": ["month", "year", "total_prior", "total_current"],
    "additionalProperties": False,
}
INSTRUCTION = ("You extract figures from one page of a City of Boulder revenue report. "
               "Copy numbers exactly as printed, without $ or commas. Use only the page.")
response_format = {"type": "json_schema", "json_schema": {"name": "totals", "strict": True, "schema": SCHEMA}}

In [ ]:
# messages: a "system" message with INSTRUCTION, then a "user" message with summary, the page
# Your code here

In [ ]:
# body: a copy of one_body; set its "messages" to messages, and its "response_format" to response_format
# (its models, temperature and provider stay as they were)
# Your code here

In [ ]:
# response: send body as before; print response.get("error")
# Your code here

In [ ]:
# Print the model that answered and whether it's in MODELS; then "usage", the tokens it read and wrote
# Your code here

In [ ]:
# content: the first choice's message's "content"; answer: json.loads(content); print answer
# Your code here

<details>
<summary>💬 Discuss</summary>

- In the key's run, the sentence came back as the text `'173908431'`, and the one-field answer as the number 173908431. Why does a check care which it is? What would `plain_text == 173908431` print?
- If a different model answered your neighbor, can you compare your two results?
- Your `usage` shows how many `reasoning_tokens` the model wrote before its answer: 3,325 of its 3,378 in the key's run. Why would they matter on a paid model?
- The free models answer 429, Too Many Requests, when they're busy. If all three did, what would `response.get("error")` print, and which cell would fail first?
- The list also passes a request on when the first model refuses it, for a misspelled `"role"`, say. If Dots answered and you expected Nemotron, how would you tell a busy model from a mistake in your `body`?

</details>

## Step 9 — Check it against week 9 (6 minutes)

Week 9 read the same table with pdfplumber, by position, and checked its sums, so we trust its figures. They're our **ground truth**: the answers we check others against. Compare each of the model's fields with them: the two totals, the year, and the month. A model that copies numbers well can still misread which year is which.

In [ ]:
# Given: week 9's figures for the last row, Total Sales and Use Tax, read with pdfplumber (chapter 9, "The Time Series"):
# for each report, the year before and the report's own year
TRUTH = {
    "revenue-report-2018": (134538892, 140648710), "revenue-report-2019": (140609751, 141954650),
    "revenue-report-2020": (141954650, 135195413), "revenue-report-2021-12": (135195413, 152302653),
    "revenue-report-2022-12": (152302653, 169112615), "revenue-report-2023-12": (169112615, 175517129),
    "revenue-report-2024-12": (175517129, 173908431),
}

In [ ]:
# prior, current: TRUTH["revenue-report-2024-12"]; print, for each total, the model's figure, week 9's, and whether they match
# Then print the model's year and month: are they 2024 and December?
# Your code here

### ⏱ Microsprint (4 minutes, alone or in pairs)

**Do:** send `body` to Dots, the third model in `MODELS`, on its own. `dots_body = dict(body)` copies `body`, as in Step 8; set the copy's `"models"` to `[MODELS[2]]`, a list of Dots alone. Then run the given cell under it, which sends it and prints the model that answered and its answer. (Gemma, the second, can't take this request: sent alone with `require_parameters`, it answers 404, "No endpoints found that can handle the requested parameters".)
**Share:** did Dots agree with your Step 8 answer, and with week 9? Which field differed, if any? If Dots answered Step 8 too, did it give the same answer twice?

In [ ]:
# Microsprint: dots_body: a copy of body (dict(body)), with "models" set to [MODELS[2]], a list of Dots alone
# Your code here

In [ ]:
# Given (microsprint): send dots_body; print any error, then the model that answered and its answer
dots = requests.post(
    f"{openrouter_base}/chat/completions",
    json=dots_body,
    headers=openrouter_headers,
).json()
print(dots.get("error"))
print(dots["model"], json.loads(dots["choices"][0]["message"]["content"]))

<details>
<summary>💬 Discuss</summary>

- If Dots and Nemotron disagreed, which was right, and which disagreement would you have caught without week 9's figures? If they agreed, does that make them right?
- `temperature` 0 asks for the most likely answer. Why might your answer differ from your neighbor's, or from one day to the next, even when the same model gives it?
- If the model had been wrong and you hadn't had week 9's figures, how would you have found out?

</details>

## Step 10 — Seven reports (14 minutes)

Now the same request for every year-end report from 2018 to 2024. A given function, `extract_totals(page, instruction)`, holds Step 8's lines: it copies `body`, puts the instruction and the page in its messages, sends it, and returns the answer as a dictionary, with the model that gave it added. Call it once by hand, on the 2024 page. Then a given loop does each report in `TRUTH`: it reads its Markdown, finds its summary page as in Step 7, calls `extract_totals()`, and compares both totals with week 9's, pausing two seconds between requests. Then add the schema to the instruction as text, as the chapter does: [`json.dumps()`](https://docs.python.org/3/library/json.html#json.dumps) writes `SCHEMA` as a string of JSON, which can join the instruction's words. A given cell runs the loop again with it. Compare the two tables. See the chapter's ["Seven Reports"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-13-ai-apis.html#seven-reports).

In [ ]:
# Given: Step 8's request as a function. extract_totals(page, instruction) copies body, puts the instruction and
# the page in its messages, sends it, and returns the answer as a dictionary, with the model that gave it added
def extract_totals(page, instruction):
    page_body = dict(body)
    page_body["messages"] = [
        {"role": "system", "content": instruction},
        {"role": "user", "content": page},
    ]
    page_response = requests.post(f"{openrouter_base}/chat/completions", json=page_body,
                                  headers=openrouter_headers).json()
    page_answer = json.loads(page_response["choices"][0]["message"]["content"])
    page_answer["model"] = page_response["model"]
    return page_answer

In [ ]:
# Call extract_totals() on summary, the 2024 page, with INSTRUCTION; print what it returns
# Your code here

In [ ]:
# Given: for each report in TRUTH, read its saved Markdown, find its summary page as in Step 7, send it with
# extract_totals() and INSTRUCTION, and check both totals against week 9's; pause 2 seconds between requests
report_rows = []
for report in TRUTH:
    report_md = requests.get(saved + report + ".md", headers=HEADERS).text
    report_pages = pd.Series(split_pages(report_md))
    report_has_total = report_pages.str.contains("Total Sales and Use Tax", regex=False)
    report_summary = report_pages[report_has_total].iloc[0]
    report_answer = extract_totals(report_summary, INSTRUCTION)
    report_answer["report"] = report
    report_answer["prior_ok"] = report_answer["total_prior"] == TRUTH[report][0]
    report_answer["current_ok"] = report_answer["total_current"] == TRUTH[report][1]
    report_rows.append(report_answer)
    time.sleep(2)
results = pd.DataFrame(report_rows).set_index("report")
print(results.drop(columns="model").to_string())
print(results["model"].value_counts())

In [ ]:
# with_schema: INSTRUCTION, then " Answer with JSON that fits this schema: ", then SCHEMA as text, with json.dumps()
# Your code here

In [ ]:
# Given: the loop above again, with with_schema, the instruction with the schema in it, in place of INSTRUCTION;
# its rows go in schema_rows, and its table in schema_results. Compare it with the first
schema_rows = []
for report in TRUTH:
    report_md = requests.get(saved + report + ".md", headers=HEADERS).text
    report_pages = pd.Series(split_pages(report_md))
    report_has_total = report_pages.str.contains("Total Sales and Use Tax", regex=False)
    report_summary = report_pages[report_has_total].iloc[0]
    report_answer = extract_totals(report_summary, with_schema)
    report_answer["report"] = report
    report_answer["prior_ok"] = report_answer["total_prior"] == TRUTH[report][0]
    report_answer["current_ok"] = report_answer["total_current"] == TRUTH[report][1]
    schema_rows.append(report_answer)
    time.sleep(2)
schema_results = pd.DataFrame(schema_rows).set_index("report")
print(schema_results.drop(columns="model").to_string())
print(schema_results["model"].value_counts())

<details>
<summary>💬 Discuss</summary>

- In the key's run, the first table got both totals right for 4 of 7 reports, 2021 to 2024, and wrong for 2018, 2019 and 2020. Print one of those reports' summary pages and read the table's last two rows: where did the model's figures come from?
- The same run called the 2019 and 2020 reports January, and the 2024 report 2025. Where on those pages could January and 2025 come from?
- The schema was in `response_format` from the start. Why would sending it again, as text in the instruction, change the answers, and would you still check them against week 9?
- The loop has no error handling. If the fifth request got a 429 from every model, or (as happened once in an earlier run) an answer with no content, so that `json.loads(None)` raised a `TypeError`, what would `report_rows` hold when the cell stopped?

</details>

## End of Part 2

On Monday, November 16, Part 3 leaves the revenue reports for Wikipedia. Embeddings measure how the opening paragraphs of five politicians' articles have changed since 2004, starting from one small request on three short sentences. If there's time, an optional last step has a model write one request to week 11's API from the tools the Census Bureau's MCP server offers. It starts with its own setup step. Friday's last 15 minutes go to your portfolio checkpoint: see the last cell of this notebook.

# Part 3 — Monday, November 16: search by meaning

In Part 2 the model read one page, and we chose the page. With a hundred passages, we first need a way to find the ones worth reading. An **embedding** turns a passage into a list of numbers, so that passages about the same thing get similar numbers.

Boulder's reports are a poor test of that. Their pages are mostly tables, and an embedding captures what a passage is about, not which figure it prints: every year's report has a page on marijuana revenue, and to an embedding those pages look nearly alike. Wikipedia suits it better. Its articles are prose, they're rewritten as their subjects' lives change, and every version is kept in the article's history (week 10). So today we take the **lead**, the paragraphs above an article's first heading, of five articles: Barack Obama, Hillary Clinton, Donald Trump, Joe Biden, and Kamala Harris. We take each lead as it stood at midnight UTC on January 1 of every year from 2004 to 2026, and as it stands now. How far was each year's lead from today's, and when did it change most?

We go one small step at a time: one old version by hand before the saved leads, and one small embedding request, three short sentences, before all 116 leads. If there's time at the end, an optional step shows **tools**.

## Step 11 — Set up, and one old version by hand (15 minutes)

Part 3 starts fresh. The setup is given: put your email in its `HEADERS` and run it, then run the next given cell, whatever the check said. It asks for your key if it's `MISSING`. Wikimedia asks for week 10's User-Agent, with the library and its version, so `HEADERS` takes that form today.

Then get one old version of Obama's lead by hand, in two requests to the Action API, one cell at a time:
1. **Ask for one revision.** [`prop=revisions`](https://www.mediawiki.org/wiki/API:Revisions) with `rvstart` set to `2009-01-01T00:00:00Z`, `rvdir=older` and `rvlimit=1` starts at that moment, walks back in time, and stops after one: the version a reader saw on New Year's Day. Its `revid` names that version.
2. **Ask for its lead.** [`action=parse`](https://www.mediawiki.org/wiki/API:Parsing_wikitext) with `oldid` set to that `revid`, `prop=text` and `section=0` returns the version's lead as HTML (section 0 is everything above the first heading; `formatversion=2` makes `text` a plain string).
3. **Read it.** Print each paragraph (`<p>`) with BeautifulSoup's [`get_text()`](https://www.crummy.com/software/BeautifulSoup/bs4/doc/#get-text). Wikipedia keeps the wikitext of old versions, not their HTML, so `parse` draws the 2009 wikitext with today's templates, and templates deleted since then come back as stray links named `Template:…`.
4. **Clean it.** Two given cells do what the chapter's `lead_text()` does. The first finds the leftovers by CSS selector (week 5) with [`select()`](https://www.crummy.com/software/BeautifulSoup/bs4/doc/#css-selectors-through-the-css-property), and `decompose()` deletes each. The second keeps each paragraph with text and joins them.
5. **Check it.** A given cell loads all the leads, collected and cleaned the same way on 2026-11-09, and we check that your 2009 lead is the same text as the saved one.

See the chapter's ["Embeddings and Retrieval"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-13-ai-apis.html#embeddings-and-retrieval).

In [ ]:
# Given: Part 3's setup. Put your email in HEADERS (week 10's form, with requests' version), then run it.
# It imports what Part 3 needs, checks your OpenRouter key, and sets the addresses (OpenRouter, Wikipedia's
# Action API, and the course's saved leads). The next cell reads the key
import os
import re
import json
import time
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup

HEADERS = {"User-Agent": f"WebDataScience/1.0 (your-email@colorado.edu) python-requests/{requests.__version__}"}
print("OPENROUTER_KEY:", "set" if os.environ.get("OPENROUTER_KEY") else "MISSING")
openrouter_base = "https://openrouter.ai/api/v1"
API = "https://en.wikipedia.org/w/api.php"
wiki = "https://raw.githubusercontent.com/cuinfoscience/INFO4617-Fall2026/main/labs/week-13/data/wikipedia/"

In [ ]:
# Given: run this whatever the check said. If the key was MISSING, it asks with getpass, in a box under this cell;
# then, either way, it reads the key, and openrouter_headers adds it to HEADERS, as on Friday
from getpass import getpass
if not os.environ.get("OPENROUTER_KEY"):
    os.environ["OPENROUTER_KEY"] = getpass("OPENROUTER_KEY isn't set. Paste it here: ")
openrouter_key = os.environ["OPENROUTER_KEY"]
openrouter_headers = {"Authorization": f"Bearer {openrouter_key}", **HEADERS}

In [ ]:
# params: one revision, the last one before 2009 began: action=query, titles="Barack Obama", prop=revisions,
# rvprop="ids|timestamp", rvlimit=1, rvstart="2009-01-01T00:00:00Z", rvdir="older", format=json
# Your code here

In [ ]:
# data: send params to API with HEADERS, as JSON; found: data["query"]["pages"]; page: its first value
# (list(found.values())[0]); revision: page["revisions"][0]. Print revision
# Your code here

In [ ]:
# params: that revision's lead, as HTML: action=parse, oldid=revision["revid"], prop=text, section=0,
# format=json, formatversion=2
# Your code here

In [ ]:
# html: send it, and take the JSON's ["parse"]["text"]; soup: BeautifulSoup(html, "html.parser")
# Print the first 150 characters of each <p>'s get_text(" ", strip=True)
# Your code here

In [ ]:
# Given: find the leftovers by CSS selector (tables, footnote marks, notes, links to missing templates,
# error messages), and delete each one from the page with decompose()
leftovers = soup.select(
    "table, .reference, .hatnote, .shortdescription, "
    "style, a.new[title^='Template:'], .error"
)
for tag in leftovers:
    tag.decompose()
print(len(leftovers), "leftovers deleted")

In [ ]:
# Given: keep the text of each paragraph that has any, join them with blank lines, and take out the space
# get_text(" ") leaves before punctuation (re.sub, a regular expression); print the lead's start
paragraphs = []
for paragraph in soup.find_all("p"):
    if paragraph.get_text(strip=True):
        text = paragraph.get_text(" ", strip=True)
        paragraphs.append(text)
joined = "\n\n".join(paragraphs)
lead = re.sub(r" ([,.;:])", r"\1", joined)
print(lead[:300])

In [ ]:
# Given: every lead, collected and cleaned this way on 2026-11-09: one row per article and year, plus each
# article's "current" version. Collecting them at one request a second drew 429s for about 30 minutes
leads = pd.read_csv(wiki + "leads.csv")
print(len(leads), "leads")
print(leads.groupby("title")["year"].count())

In [ ]:
# saved_lead: the "lead" of leads' row for "Barack Obama" and year "2009"; print whether lead == saved_lead
# Your code here

<details>
<summary>💬 Discuss</summary>

- Collecting these leads at one request a second drew 429 (Too Many Requests) for about 30 minutes, with waits of up to 40 seconds. If all of us ran that loop at once from this room, whose requests would Wikipedia refuse, and why does week 10's User-Agent matter here?
- Our two requests assume they worked. Wikimedia's 429 is plain text, not JSON (week 10): which line would fail, and what would it say?
- `parse` drew the 2009 version with today's templates, and our cleanup took out what it could see was broken. What might still come from today rather than from 2009, without looking broken?

</details>

## Step 12 — One small embedding, then every lead (20 minutes)

[OpenRouter's embeddings endpoint](https://openrouter.ai/docs/api/reference/embeddings) takes a `model` and an `input`, a list of texts, and returns one **vector** per text: a list of numbers, the text's embedding. Before we embed 116 leads, we make **one small request** with three short sentences, and look at what comes back. Two of them say the same thing in different words, and the third is about something else. If embeddings work as promised, the first two should be closer to each other than either is to the third.

We use one free model, `EMBEDDER`, for every vector. Two models turn the same text into different numbers, so their vectors can't be compared, and an embedding model never gets a fallback list. To compare two vectors we use **cosine similarity**, which measures how nearly they point the same way: 1 if exactly, less the further apart. A given cell scales each vector to length 1 (here, a vector's length is the square root of its squared numbers added up, not how many numbers it has). For vectors of length 1, cosine similarity is their **dot product**, `@`: multiply the two lists number by number, and add up.

Then the same idea, at full size. The model reads at most about 4,096 tokens of a text, which is why we embed the lead and not the article. A given cell embeds all 116 leads, 20 to a request; if the free model's daily limit stops it, the next cell loads the same vectors, saved on 2026-11-09. A given cell compares each lead with its article's current lead; we check that each current lead scores 1 against itself, and [`pivot()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.pivot.html) the result into a table with a row per year and a column per article, which a given cell plots. See the chapter's ["Embedding and Cosine Similarity"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-13-ai-apis.html#embedding-and-cosine-similarity).

In [ ]:
# EMBEDDER = "nvidia/nemotron-3-embed-1b:free"; sentences: a list of three short sentences, two that say the
# same thing in different words, and one about something else
# Your code here

In [ ]:
# small: the JSON of one POST to f"{openrouter_base}/embeddings", with json {"model": EMBEDDER, "input": sentences}
# and headers=openrouter_headers. Print how many vectors came back, how many numbers the first holds, and its "usage"
# Your code here

In [ ]:
# Given: the three vectors as the rows of a NumPy array, each scaled to length 1 (np.linalg.norm gives each row's
# length; axis=1 measures rows, keepdims=True lets each row divide by its own)
three = np.array([item["embedding"] for item in small["data"]])
three = three / np.linalg.norm(three, axis=1, keepdims=True)
print(three.shape)

In [ ]:
# Print the similarity of sentences 0 and 1 (three[0] @ three[1]), and of sentences 0 and 2, each rounded to 3 places
# Your code here

In [ ]:
# Given: embed every lead, 20 to a request, with a pause between requests; vectors: one row per lead, scaled to length 1
vectors = []
for start in range(0, len(leads), 20):
    batch = leads["lead"][start:start + 20].tolist()
    response = requests.post(f"{openrouter_base}/embeddings", json={"model": EMBEDDER, "input": batch},
                             headers=openrouter_headers).json()
    vectors += [item["embedding"] for item in response["data"]]
    time.sleep(1)
vectors = np.array(vectors)
vectors = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)
print(vectors.shape)

In [ ]:
# Given, only if the cell above stopped: remove the "#" from the next two lines, and run this cell. It loads the
# vectors saved on 2026-11-09, from the same model, already scaled to length 1, in the same order as leads
# import io
# vectors = np.load(io.BytesIO(requests.get(wiki + "lead-vectors.npy", headers=HEADERS).content))

In [ ]:
# Given: each article's current vector (row i of vectors is row i of leads); then each lead's similarity to its own
# article's current lead, one per row (a list comprehension: the loop written inside the brackets)
today = {}
for i, row in leads.iterrows():
    if row["year"] == "current":
        today[row["title"]] = vectors[i]
leads["similarity"] = [
    vectors[i] @ today[title]
    for i, title in enumerate(leads["title"])
]

In [ ]:
# current: leads' rows whose year is "current"; print their "title" and "similarity", each of which should be 1
# Your code here

In [ ]:
# past: the leads whose year isn't "current"; table: past pivoted to a row per year and a column per title,
# with similarity as values. Make its index (the years) integers; print it, rounded to 3 places
# Your code here

In [ ]:
# Given: plot table, a line per article with a marker at each year, titled, with both axes and the legend labeled
fig, ax = plt.subplots(figsize=(8, 4.5))
table.plot(ax=ax, marker="o")
ax.set_title("How close each January 1 lead is to the article's current lead")
ax.set_xlabel("Year (the lead as of January 1, 00:00 UTC)")
ax.set_ylabel("Cosine similarity with the current lead")
ax.legend(title="Article")
plt.show()

### ⏱ Microsprint (6 minutes, alone or in pairs)

Each line climbs toward 1, but not evenly. A jump is a year whose lead moved much closer to today's than the year before's had.

**Do:** pick one article. [`diff()`](https://pandas.pydata.org/docs/reference/api/pandas.Series.diff.html) gives each year's change from the year before, and `idxmax()` the year of the biggest rise. Print that year and the size of its jump. Then, in the next cell, print the first 400 characters of that year's lead and of the year before's.
**Share:** the article, the year, the jump, and what the lead said differently: a new office, or a rewrite?

In [ ]:
# Microsprint: title: your article; jumps: table[title].diff(); year: jumps.idxmax(); print year and the jump, rounded to 3
# Your code here

In [ ]:
# For the year before and that year, print the first 400 characters of the lead (leads' "year" holds text: use str())
# Your code here

<details>
<summary>💬 Discuss</summary>

- Harris's lead jumped most in 2021 (0.114), when it first called her vice president-elect, and Biden's in 2009, for the same reason. Trump's biggest jump came in 2009, not 2017: his lead grew from 93 words to 353 (the `words` column), with no new office. What besides events can move a lead closer to today's?
- Clinton's lead went from 0.943 in 2024 to 0.992 in 2025, with no new office: editors rewrote her opening. What does a similarity measure, and which claims about how Wikipedia described these people can it not support?
- The given loop assumes every request worked. If the free model answered 429 to the fourth batch, the cell would stop at `response["data"]`: what would `vectors` hold, and what does the saved vectors' cell spare you?

</details>

## Step 13 — What did we learn? (3 minutes)

In a few sentences, as comments:

- Datalab read a picture of a table and wrote descriptions of charts. Which of its output would you publish, and which would you check first?
- In Part 2, which reports did the model get wrong, and was the problem the model or what we sent it?
- Step 12 put a number on how far each year's lead was from today's. Which of its jumps could you explain only by reading the leads, and what can a similarity not tell you?
- Step 12 tried three short sentences before 116 leads. What could you check on that small request that you couldn't have checked on the full run?
- Every answer this week was checked: against week 9's figures, a sum, or the leads themselves. What would you check your own project's model outputs against?

**Recommended Exercises ahead** (in the chapter's companion notebook, due Wednesday, November 18, at 11:59 p.m.): how alike two texts are, for two Wikipedia articles or two public PDFs you choose, with no private individuals' names.

- Step 1, choose your two texts and check your key or keys: Step 0's check, which never prints a key.
- Step 2, get the first text: for an article, Step 11's two requests (leave out `rvstart` to get the latest revision) and the cleanup the chapter's `lead_text()` gathers; for a PDF, Monday's Steps 1–3 and 5, the lines the chapter's `convert()` gathers, and one page. For a PDF on your computer rather than on the web, the chapter's "Submitting a Document" says how to upload it.
- Step 3, embed it: Step 12's request to the embeddings endpoint, the chapter's `embed_text()`.
- Step 4, a second text and the two texts' similarity: your Step 2 code as a function, and Step 12's dot product.
- Step 5, a third text you expect to be unlike both, as a baseline: the same function, and the similarity of each pair.
- Step 6, report: what the similarities say, what reading the texts shows that the number can't, and how many requests the run made.

In [ ]:
# Your answers here, as comments


## Step 14 (optional, if time) — Tools: a model writes a Census request, and skills (10 minutes)

**This step is optional:** do it in class if there's time, or on your own. It needs week 11's `CENSUS_KEY`, which its first given cell asks for, and Friday's `MODELS`, which that cell sets again.

A model can't fetch data: all it can do is write text. But a request is text, so if a program tells a model which requests it can make, the model can write one, and the program sends it. Each request a model may make is a **tool**: a name, a description, and its inputs, described by a JSON Schema like Part 2's. The **Model Context Protocol** (MCP) is a shared way for a server to tell a program which tools it offers: the program sends the method [`tools/list`](https://modelcontextprotocol.io/specification/2025-06-18/server/tools#listing-tools), and the server answers with each tool's `name`, `description`, and `inputSchema`. The Census Bureau's [MCP server](https://github.com/uscensusbureau/us-census-bureau-data-api-mcp) offers five tools for week 11's API. It runs on your own computer, with Docker or Node, so we don't install it: it was run once, on 2026-11-09, and its answers were saved in this course's repository, in `labs/week-13/data/census-mcp/`. Two given cells read its tools from there and put them in the form OpenRouter's [tool calling](https://openrouter.ai/docs/guides/features/tool-calling) asks for: each tool's `inputSchema` becomes its `parameters`. We send them to a model with a week 11 question, and [`tool_choice`](https://openrouter.ai/docs/guides/features/tool-calling#tool-choice-configuration) `"auto"`, OpenRouter's default, lets the model choose between a tool call and text. The chapter sends `"required"`, OpenAI's value for a call and no text, which OpenRouter's documentation doesn't list; on 2026-11-09 Nemotron honored it, but Dots, the last of `MODELS`, answered it with garbled arguments or no call at all, and with `"auto"` it wrote the right call three times in five. With `"auto"`, a model may answer in prose instead, with no `tool_calls`; when it does call a tool, the call comes back in the message's `tool_calls`, with its `arguments` as a JSON string. The model only writes the call. A given cell runs it, as the server would have: it reads the call's arguments with `json.loads()`, prints whether the model chose `fetch-aggregate-data`, the one tool our code runs, as the chapter checks the call before running it; then turns the arguments into week 11's request, sends it with `census_key`, and prints the answer, never the URL, which holds your key. We check that answer against week 11's figure, \$103,994 with a margin of error of \$2,268, and against what the real server returned. Last, a **skill**, plainer still: a folder of instructions in Markdown, whose `name` and `description` tell a model when to read the rest. A given cell reads the first lines of a public one, for PDFs, and its headings. See the chapter's ["Tools, MCP, and Skills"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-13-ai-apis.html#tools-mcp-and-skills), and its ["Running the Call Yourself"](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-13-ai-apis.html#running-the-call-yourself) for the full check.

In [ ]:
# Given (optional step): read week 11's CENSUS_KEY, asking with getpass if it isn't set; and Friday's MODELS,
# three free models in order
if not os.environ.get("CENSUS_KEY"):
    os.environ["CENSUS_KEY"] = getpass("CENSUS_KEY isn't set. Paste it here: ")
census_key = os.environ["CENSUS_KEY"]
MODELS = ["nvidia/nemotron-3.5-super-vl:free", "google/gemma-4-31b-it:free", "dots-studio/dots-3-note-preview:free"]

In [ ]:
# Given: the Census MCP server's answer to tools/list, saved in the course's repository; print each tool's name
# and the inputs its JSON Schema requires
cmcp = "https://raw.githubusercontent.com/cuinfoscience/INFO4617-Fall2026/main/labs/week-13/data/census-mcp/"
tools = requests.get(cmcp + "tools.json", headers=HEADERS).json()
for tool in tools:
    print(tool["name"], tool["inputSchema"]["required"])

In [ ]:
# Given: each MCP tool in OpenRouter's form, its inputSchema as "parameters"; the instruction; and week 11's question,
# with the FIPS codes week 11 used
functions = [{"type": "function", "function": {"name": tool["name"], "description": tool["description"],
                                               "parameters": tool["inputSchema"]}} for tool in tools]
TOOL_INSTRUCTION = "Choose one tool call that answers the question. Use the tools' parameters exactly."
question = ("What was the median household income of Boulder County, Colorado (state FIPS 08, county FIPS 013), "
            "in the 2024 ACS 5-year estimates, table B19013?")

In [ ]:
# body: MODELS, temperature 0, "tools": functions, "tool_choice": "auto", messages (system TOOL_INSTRUCTION, user question)
# Send it as on Friday, with headers=openrouter_headers; call: the first choice's message's "tool_calls"[0]["function"]
# Print the model that answered, call["name"], and call["arguments"], a string of JSON
# Your code here

In [ ]:
# Given: run the call ourselves. args: the call's arguments, from JSON to a dictionary, read here so that this cell
# always runs the call it checks. Is it the one tool this code runs? Then url: week 11's address, from args' year
# and dataset; get: group(...) for a whole table, or the variables joined with commas; send it with census_key,
# and print data, never the URL: your key is in it
args = json.loads(call["arguments"])
print(call["name"], call["name"] == "fetch-aggregate-data")
url = f"https://api.census.gov/data/{args['year']}/{args['dataset']}"
if "group" in args["get"]:
    get = f"group({args['get']['group']})"
else:
    get = ",".join(args["get"]["variables"])
params = {"get": get, "for": args["for"], "in": args["in"], "key": census_key}
data = requests.get(url, params=params, headers=HEADERS).json()
print(data)

In [ ]:
# row: dict(zip()) of data's names (row 0) and values (row 1); print its B19013_001E, whether it's week 11's "103994",
# and its margin of error, row.get("B19013_001M"); then print the text of session.json's last message, the real server's answer
# Your code here

In [ ]:
# Given: a public skill's SKILL.md, for PDFs: its first 5 lines (the front matter: name and description),
# then its "### " headings, the tools it teaches
skill = requests.get("https://raw.githubusercontent.com/anthropics/skills/main/skills/pdf/SKILL.md", headers=HEADERS).text
print("\n".join(skill.splitlines()[:5]))
print([line for line in skill.splitlines() if line.startswith("### ")])

### ⏱ Microsprint (3 minutes, alone or in pairs)

**Do:** ask as a person would, by name and without FIPS codes: "What was the median household income of the city of Boulder, Colorado, in the 2024 ACS 5-year estimates, table B19013?" Put it in `body`'s user message, send it, and print the tool call.
**Share:** the tool the model chose, and its arguments. Could our cell that runs the call run this one? What would you send instead, with week 11's place code for Boulder, 07850?

In [ ]:
# Microsprint: put the question above in body's user message (body["messages"][1]["content"]), and send it
# call: the first tool call's "function", as above; print the model, call["name"], and json.loads() of its "arguments"
# Your code here

<details>
<summary>💬 Discuss</summary>

- In the key's run, the microsprint's model chose `resolve-geography-fips`, a tool we haven't written, which the real server answers from a database it builds when it's installed. What would our cell that runs the call do with it, and what should a program do with a call it can't run?
- Our cell checked the tool's name, but the model's arguments went into our request unchecked. In an earlier run the model also sent `descriptive`, and our cell quietly dropped it. Who should check them before they're sent: the model, the schema, our code, or the Census API? In week 11 the API answered 400 to a place without a state.
- MCP standardizes how tools are described, so any model can read the Census server's. It doesn't make the answer right: what did we check ours against? In the key's run the model asked for the whole table, `group` B19013, margin of error included; in an earlier run it asked for the estimate alone, `B19013_001E`, and the margin printed `None`. Would we have noticed the missing margin without week 11's figure?

</details>

## Friday, November 13: your portfolio checkpoint (15 minutes)

Friday's last 15 minutes, after Part 2, go to your project. This week's checkpoint, **Analyze your data**, is due Sunday, November 15:

- **An analysis notebook** that runs top to bottom, with its output visible;
- **results that answer at least part of your question**: the counts or comparisons your proposal named;
- **at least one figure** saved in your repository, with a title, labeled axes, and a one-sentence caption;
- **the README's Methods section** and your preliminary **Findings**.

Use the time to run your analysis notebook once, top to bottom, here, where you can ask. If you use a model anywhere in your analysis, say which, when, and what you checked its output against, in Methods.

The lab's Recommended Exercises are due three days after the checkpoint, on Wednesday, November 18, at 11:59 p.m.